# 02 — Dataset Preparation

Validates, formats, and splits the raw generated conversations into  
`data/dataset_train.jsonl` and `data/dataset_val.jsonl` ready for fine-tuning.

## What this does
1. Loads all conversations from `data/raw_conversations.jsonl`
2. Re-runs validation — filters out hard failures (wrong language, too short, etc.)
3. Applies the Qwen ChatML format (prepends system prompt, keeps `[CONTEXT RAG]` as a system message)
4. Saves an 90/10 train/val split
5. Prints statistics and example entries

In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FINE_TUNING_DIR  = REPO_ROOT / "fine_tuning"
RAW_FILE         = FINE_TUNING_DIR / "data" / "raw_conversations.jsonl"
CITY_INDEX_PATH  = FINE_TUNING_DIR / "configs" / "city_index.json"
DATA_DIR         = FINE_TUNING_DIR / "data"

print(f"Raw file : {RAW_FILE} — exists: {RAW_FILE.exists()}")

## Step 1 — Load raw conversations

In [ ]:
from fine_tuning.src.dataset_formatter import load_raw_conversations

raw_conversations = load_raw_conversations(RAW_FILE)
print(f"Loaded {len(raw_conversations)} raw conversations")

## Step 2 — Validate and filter

In [ ]:
from fine_tuning.src.city_loader import load_saved_city_index
from fine_tuning.src.conversation_validator import validate_conversation, ValidationError

city_index = load_saved_city_index(CITY_INDEX_PATH)

valid        = []
soft_warning = []
hard_fail    = []

for conv in raw_conversations:
    try:
        warnings = validate_conversation(conv, city_index)
        valid.append(conv)
        if warnings:
            soft_warning.append((conv, warnings))
    except ValidationError as exc:
        hard_fail.append((conv, str(exc)))

print(f"Valid            : {len(valid)}")
print(f"  of which warned: {len(soft_warning)}")
print(f"Hard failures    : {len(hard_fail)} (excluded from training)")

if hard_fail:
    print("\nSample hard failures:")
    for conv, reason in hard_fail[:3]:
        persona = conv.get("metadata", {}).get("persona_type", "?")
        print(f"  [{persona}] {reason}")

## Step 3 — Split and save

In [ ]:
from fine_tuning.src.dataset_formatter import split_and_save

train_path, val_path = split_and_save(valid, DATA_DIR, val_fraction=0.10, seed=42)

print(f"\nTrain: {train_path}")
print(f"Val  : {val_path}")

## Step 4 — Inspect a training example

In [ ]:
# Load the first training example and pretty-print it
first_line = train_path.read_text(encoding="utf-8").splitlines()[0]
example    = json.loads(first_line)

print(f"Persona: {example['metadata'].get('persona_type')}")
print(f"Messages: {len(example['messages'])}")
print()
for msg in example["messages"]:
    role    = msg["role"].upper()
    content = msg["content"]
    preview = content[:300].replace("\n", " ")
    print(f"[{role}]")
    print(f"  {preview}..." if len(content) > 300 else f"  {content}")
    print()

## Step 5 — Dataset statistics

In [ ]:
from collections import Counter

all_examples = []
for path in [train_path, val_path]:
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            all_examples.append(json.loads(line))

persona_counts = Counter(ex["metadata"].get("persona_type") for ex in all_examples)
total_messages  = sum(len(ex["messages"]) for ex in all_examples)
total_chars     = sum(
    len(msg["content"])
    for ex in all_examples
    for msg in ex["messages"]
)

print(f"Total examples  : {len(all_examples)}")
print(f"Total messages  : {total_messages}")
print(f"Total chars     : {total_chars:,}")
print(f"Avg msgs/example: {total_messages / len(all_examples):.1f}")
print()
print("Per-persona distribution:")
for persona, count in sorted(persona_counts.items()):
    print(f"  {persona:<35}: {count}")

## Step 6 — Verify Qwen tokenization (optional but recommended)

In [ ]:
# Run this cell only if transformers is installed and you want to
# verify the ChatML format and response template masking.

try:
    from transformers import AutoTokenizer
    from fine_tuning.configs.lora_config import DEFAULT_MODEL_ID, QWEN_RESPONSE_TEMPLATE

    print(f"Loading tokenizer for {DEFAULT_MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL_ID, trust_remote_code=True)

    # Apply chat template to the first example
    messages = example["messages"]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    print("ChatML output (first 600 chars):")
    print(formatted[:600])

    print(f"\nResponse template '{QWEN_RESPONSE_TEMPLATE!r}' present: "
          f"{QWEN_RESPONSE_TEMPLATE in formatted}")

    # Tokenize and count
    tokens = tokenizer(formatted, return_tensors="pt")
    print(f"Token count: {tokens['input_ids'].shape[1]}  (max_seq_length: 2048)")

except ImportError:
    print("transformers not installed — skipping tokenization check.")
except Exception as exc:
    print(f"Could not load tokenizer: {exc}")